In [1]:
from petrel_client.client import Client
c = Client()

In [ ]:
import numpy as np
from PIL import Image

def load_mask_from_png(image_path, mask_num):
    # 读取PNG图像
    img = Image.open(image_path)
    
    # 转换为numpy数组
    mask = np.array(img)
    return mask == mask_num

# 示例调用
mask = load_mask_from_png('pnorm2:s3://mevis/val_u/JPEGImages/d56a6ec78cfa/00000.jpg', 1)
print(mask.shape)  # 输出mask的形状

In [ ]:
from tqdm import tqdm
import numpy as np
import cv2
for dd in tqdm(c.list('pnorm2:s3://mevis/videos/')):
    print(dd)
#     pnorm2:s3://mevis/val_u/JPEGImages/d56a6ec78cfa
img_bytes = c.get('pnorm2:s3://mevis/videos/d56a6ec78cfa.mp4')
img_np = np.frombuffer(img_bytes, np.uint8)
img = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
img.shape

In [ ]:
mask_dict =eval( c.get('pnorm2:s3://mevis/masks_val_u/16.json').decode().replace('null', 'None') )
len(mask_dict)
len(mask_dict['gt_masks'])
mask_dict['gt_masks'][0]

In [ ]:
dd = c.get('pnorm2:s3://mevis/val_u/meta_expressions.json')

In [ ]:
len(eval(dd.decode())['videos'])

In [ ]:
videos = eval(dd.decode())['videos']
num = 0
final = []
for k in (videos):
    for exp_id in videos[k]['expressions']:
            print(videos[k])
            break
#         video_path = 'pnorm2:s3://mevis/videos/' + k + '.mp4'
#         exp = videos[k]['expressions'][exp_id]['exp']
#         mask = 'pnorm2:s3://mevis/new_mask/' + str(len(final)) + '.json'
#         final.append({'video_path': video_path, 'exp':exp, 'mask':mask})
#     print(k)
#     num += len(videos[k]['expressions'])
#     print(videos[k])
#     break
# num

In [ ]:

import os
import time
import argparse
import cv2
import json
import numpy as np
from pycocotools import mask as cocomask
import multiprocessing as mp
import math
import pycocotools.mask as mask_util
def save_mask(mask_id, mask_path):
    path = 'pnorm2:s3://mevis/masks_val_u/' + str(mask_id[0]) + '.json'
    gt_masks = eval(c.get(path).decode())['gt_masks']
    print(len(gt_masks))
    for mask in gt_masks:
        if mask is not None:
            none_mask = mask_util.decode(mask)
            # print(none_mask.shape)
            none_mask = np.zeros_like(none_mask)
            none_mask = mask_util.encode(none_mask)
            # print(none_mask)
            break
    gt_masks_replace = []
    for mask in gt_masks:
        if mask is None:
            gt_masks_replace.append(none_mask)
        else:
            gt_masks_replace.append(mask)

    fin_masks = {'gt_mask': gt_masks_replace}
    c.put( mask_path, str(fin_masks) )
        
def save_mul_mask(mask_id, mask_path):
    memory_mask = None
    for now_id in range(len(mask_id)):
        path = 'pnorm2:s3://mevis/masks_val_u/' + str(mask_id[now_id]) + '.json'
        gt_masks = eval(c.get(path).decode())['gt_masks']
        for mask in gt_masks:
            if mask is not None:
                none_mask = mask_util.decode(mask)
                # print(none_mask.shape)
                none_mask = np.zeros_like(none_mask)
                none_mask = mask_util.encode(none_mask)
                # print(none_mask)
                break
        gt_masks_replace = []
        for mask in gt_masks:
            if mask is None:
                gt_masks_replace.append(none_mask)
            else:
                gt_masks_replace.append(mask)
        if memory_mask is None:
            memory_mask = mask_util.decode(gt_masks_replace)
#             print(type(memory_mask))
#             print(memory_mask)
        else:
            memory_mask = np.bitwise_or(memory_mask, mask_util.decode(gt_masks_replace))

            
    fin_masks = {'gt_mask': mask_util.encode(memory_mask)}
    c.put( mask_path, str(fin_masks) )

In [ ]:
videos = eval(dd.decode())['videos']
num = 0
final = []
for k in (videos):
    for exp_id in videos[k]['expressions']:
#         video_path = 'pnorm2:s3://mevis/videos/' + k + '.mp4'
#         exp = videos[k]['expressions'][exp_id]['exp']
        mask = 'pnorm2:s3://mevis/new_mask1/' + str(len(final)) + '.json'
        mask_id = videos[k]['expressions'][exp_id]['anno_id']
        if len(mask_id) == 1:
            save_mask(mask_id, mask)
        else:
            save_mul_mask(mask_id, mask)
        final.append(1)
#     break

In [ ]:
import json
with open('/mnt/petrelfs/yanziang/trackingllm/videoagent/mask_dict.json', 'r') as f:
    mask_dict2 = json.load(f)
mask_dict2.keys()
len(mask_dict2['0'])

In [ ]:
len(final)

In [ ]:
gt_masks = c.get('pnorm2:s3://mevis/masks_val_u/0.json')
len(eval(gt_masks.decode())["gt_masks"])
eval(gt_masks.decode())["gt_masks"][0]

In [ ]:
len(eval(gt_masks)['gt_masks']) 

In [ ]:
with open('mevis_val_u3.json', 'r') as f:
    data = json.load(f)

final_data = []
for dd in data:
    dd['video_path'] = 'pnorm2:s3://mevis/val_u/JPEGImages/' + dd['video_path'].split('.')[0].split('/')[-1]
    final_data.append(dd)

with open('mevis_val_u4.json', 'w') as f:
    json.dump(final_data, f)

In [ ]:
for dd in tqdm(c.list('pnorm2:s3://ava/frames/clip_850to1850/-5KQ66BBWC4/')):
    print(dd)

In [ ]:
import json
path = '/mnt/petrelfs/share_data/yanziang/llava_video/0_30_s_academic_mc_v0_1_qa_processed.json'

with open(path,'r') as f:
    data = json.load(f)
print(len(data))

In [ ]:
num =  0
for dd in data:
    if 'youtube' not in dd['video']:
        num += 1
print(data[0])

In [1]:
from petrel_client.client import Client
c = Client()

In [20]:
all_d = []
for dd in c.list('pnorm2:s3://svit/VG_100K_2/'):
    all_d.append(dd)

In [21]:
len(all_d)

43903

In [23]:
all_d[10000]

'2386177.jpg'